# Uso de script de modularización

### Paso 1 — Carga del panel de ventanas electorales y Revisión de variables



In [ ]:
import sys
import pandas as pd

general_path = "/workspaces/analisis-politica-economia/"
data_path = f"{general_path}data/tfi_data/"
sys.path.insert(0, f"{general_path}/src")  
import ml_models
from ml_models.cargar_panel import cargar_panel 
import importlib
import ml_models.lasso
from ml_models.lasso import *
importlib.reload(ml_models.lasso)

NIVELES = ["municipal", "provincial", "nacional"]
paneles = {nivel: cargar_panel(nivel,f"{data_path}panel_ventanas.csv") for nivel in NIVELES}

for nivel, df in paneles.items():
    print(f"{nivel}: {df.shape[0]} filas x {df.shape[1]} columnas")

municipal: 12 filas x 145 columnas
provincial: 12 filas x 145 columnas
nacional: 7 filas x 145 columnas


In [24]:
cols_vc_por_nivel = {}
corr_por_nivel = {}
for nivel in NIVELES: 
    df = paneles[nivel]
    cols_vc = [
        c for c in df.columns
        if c.endswith("_vc") and pd.api.types.is_numeric_dtype(df[c])
    ]

    corr = df[cols_vc].corr(method="pearson")  # pairwise, ignora NaN automáticamente
    cols_vc_por_nivel[nivel] = cols_vc
    corr_por_nivel[nivel] = corr
for nivel in NIVELES:
    print(f"{nivel}: {len(cols_vc)} variables _vc, N={len(df)}")
corr_por_nivel["municipal"] 

municipal: 37 variables _vc, N=7
provincial: 37 variables _vc, N=7
nacional: 37 variables _vc, N=7


,desocupacion_final_vc,desocupacion_nivel_vc,desocupacion_pendiente_vc,desocupacion_volatilidad_vc,emae_final_vc,emae_nivel_vc,emae_pendiente_vc,emae_volatilidad_vc,icc_final_vc,icc_nivel_vc,...,resultado_fiscal_pendiente_vc,resultado_fiscal_volatilidad_vc,salario_real_final_vc,salario_real_nivel_vc,salario_real_pendiente_vc,salario_real_volatilidad_vc,tc_oficial_final_vc,tc_oficial_nivel_vc,tc_oficial_pendiente_vc,tc_oficial_volatilidad_vc
desocupacion_final_vc,1.000000,0.967016,-0.892786,0.485517,-0.740469,-0.704289,-0.067340,0.035119,-0.367137,-0.418511,...,0.038489,-0.227004,-0.388741,-0.308488,-0.273736,-0.141247,-0.219954,-0.209513,-0.233651,-0.231804
desocupacion_nivel_vc,0.967016,1.000000,-0.919701,0.663285,-0.787546,-0.852487,0.264270,0.132324,-0.285541,-0.301427,...,0.017468,-0.233228,-0.424350,-0.345234,-0.260197,-0.253594,-0.266020,-0.255274,-0.281650,-0.279971
desocupacion_pendiente_vc,-0.892786,-0.919701,1.000000,-0.456562,0.531637,0.726332,-0.688258,-0.328348,0.188713,0.327325,...,0.109835,0.341397,0.440421,0.399307,0.119471,0.375330,0.210948,0.207726,0.213888,0.213485
desocupacion_volatilidad_vc,0.485517,0.663285,-0.456562,1.000000,-0.649763,-0.754252,0.389727,0.063731,-0.146715,0.037211,...,0.229329,0.118886,-0.178111,-0.107868,-0.246661,-0.142980,-0.138704,-0.125492,-0.164768,-0.163047
emae_final_vc,-0.740469,-0.787546,0.531637,-0.649763,1.000000,0.937717,-0.022200,-0.139702,0.083070,-0.380826,...,0.266516,0.360963,0.628997,0.524213,0.139652,0.445185,0.339913,0.333188,0.349312,0.348770
emae_nivel_vc,-0.704289,-0.852487,0.726332,-0.754252,0.937717,1.000000,-0.358049,-0.331488,-0.156203,-0.593617,...,0.317018,0.417480,0.682740,0.594128,0.016899,0.489619,0.324931,0.311798,0.347100,0.345560
emae_pendiente_vc,-0.067340,0.264270,-0.688258,0.389727,-0.022200,-0.358049,1.000000,0.524923,0.590606,0.645797,...,-0.257539,-0.224941,-0.312088,-0.327862,0.446168,-0.229857,0.094309,0.110495,0.060493,0.062829
emae_volatilidad_vc,0.035119,0.132324,-0.328348,0.063731,-0.139702,-0.331488,0.524923,1.000000,0.632677,0.638326,...,-0.452748,-0.500234,-0.512624,-0.522912,0.092627,-0.476295,-0.502385,-0.481152,-0.538529,-0.535687
icc_final_vc,-0.367137,-0.285541,0.188713,-0.146715,0.083070,-0.156203,0.590606,0.632677,1.000000,0.755668,...,-0.430499,-0.543213,-0.346358,-0.395716,0.262404,-0.345966,-0.197192,-0.189837,-0.208397,-0.207307
icc_nivel_vc,-0.418511,-0.301427,0.327325,0.037211,-0.380826,-0.593617,0.645797,0.638326,0.755668,1.000000,...,-0.561208,-0.498836,-0.398036,-0.431806,0.409915,-0.401327,-0.235578,-0.220806,-0.261492,-0.259385


### Paso 2 — Sub-selección: colapsar clusters redundantes

La matriz de correlación mostró un cluster de colinealidad casi perfecta entre `ipc_*` y `tc_oficial_*` (r > 0.98 en todo el bloque), además de pares menores en `desocupacion`, `resultado_fiscal` y `salario_real`. Se busca simplificar reduciendo los datos que son redundantes para evitar que LASSO elija de forma aleatoria entre esas opciones.

**Parametrización fijada:**

| Decisión | Valor |
|---|---|
| Umbral de redundancia | `\|r\| ≥ 0.90` |
| Alcance | Transitivo (single-linkage): si A-B≥0.90 y B-C≥0.90, A/B/C van al mismo cluster aunque A-C no llegue al umbral |
| Desempate 1 | Sufijo `_nivel_vc` preferido sobre `_final`/`_pendiente`/`_volatilidad`/`_acum` |
| Desempate 2 | Prioridad teórica de la variable (`ipc` > `desocupacion` > `icg` > `icc` > `salario_real` > `tc_oficial` > `reservas` > `resultado_fiscal`) — orden de relevancia en la literatura de voto económico citada, no un criterio estadístico |

In [25]:
UMBRAL_REDUNDANCIA = 0.90
PRIORIDAD_TEORICA = ["ipc", "desocupacion", "icg", "icc", "salario_real", "tc_oficial", "reservas", "resultado_fiscal", "emae"]
ORDEN_SUFIJO = ["_nivel_vc", "_final_vc", "_pendiente_vc", "_volatilidad_vc", "_acum_vc"]

clusters_por_nivel = {}
columnas_finales_por_nivel = {}

for nivel in NIVELES:
    df = paneles[nivel]
    cols_vc = cols_vc_por_nivel[nivel]
    corr = corr_por_nivel[nivel]
    print(f"\n\nNivel: {nivel}")
    clusters_por_nivel[nivel] = encontrar_redundantes(corr, UMBRAL_REDUNDANCIA)
    columnas_finales_por_nivel[nivel] = [elegir_representante(cl, df, ORDEN_SUFIJO, PRIORIDAD_TEORICA) if len(cl) > 1 else next(iter(cl)) for cl in clusters_por_nivel[nivel]]

    print(f"De {len(cols_vc)} columnas _vc, quedan {len(columnas_finales_por_nivel[nivel])} tras colapsar clusters (umbral={UMBRAL_REDUNDANCIA})\n")
    for cl in clusters_por_nivel[nivel]:
        if len(cl) > 1:
            elegido = elegir_representante(cl, df, ORDEN_SUFIJO, PRIORIDAD_TEORICA)
            print(f"cluster ({len(cl)}): {sorted(cl)} -> queda: {elegido}")



Nivel: municipal
De 37 columnas _vc, quedan 24 tras colapsar clusters (umbral=0.9)

cluster (3): ['desocupacion_final_vc', 'desocupacion_nivel_vc', 'desocupacion_pendiente_vc'] -> queda: desocupacion_nivel_vc
cluster (2): ['emae_final_vc', 'emae_nivel_vc'] -> queda: emae_nivel_vc
cluster (8): ['ipc_final_vc', 'ipc_nivel_vc', 'ipc_pendiente_vc', 'ipc_volatilidad_vc', 'tc_oficial_final_vc', 'tc_oficial_nivel_vc', 'tc_oficial_pendiente_vc', 'tc_oficial_volatilidad_vc'] -> queda: ipc_nivel_vc
cluster (2): ['resultado_fiscal_final_vc', 'resultado_fiscal_nivel_vc'] -> queda: resultado_fiscal_nivel_vc
cluster (3): ['resultado_fiscal_volatilidad_vc', 'salario_real_final_vc', 'salario_real_nivel_vc'] -> queda: salario_real_nivel_vc


Nivel: provincial
De 37 columnas _vc, quedan 24 tras colapsar clusters (umbral=0.9)

cluster (3): ['desocupacion_final_vc', 'desocupacion_nivel_vc', 'desocupacion_pendiente_vc'] -> queda: desocupacion_nivel_vc
cluster (2): ['emae_final_vc', 'emae_nivel_vc'] -> qu

In [26]:
datos_final = {}
for nivel in NIVELES:
    X, y = construir_Xy_final(nivel, columnas_finales_por_nivel[nivel], paneles, target="delta_voto_exit_total_pct")
    datos_final[nivel] = (X, y)
    print(f"{nivel}: N={len(y)}, P={X.shape[1]}")

[municipal] excluye 1 fila(s) por NaN: ['municipal_2001_2003']
municipal: N=11, P=24
[provincial] excluye 1 fila(s) por NaN: ['provincial_2001_2003']
provincial: N=11, P=24
[nacional] excluye 1 fila(s) por NaN: ['nacional_2013_2015']
nacional: N=6, P=25


In [27]:
faltantes = columnas_nan("nacional", "nacional_2013_2015", columnas_finales_por_nivel["nacional"], paneles)
print("Columna(s) que rompen nacional_2013_2015:", faltantes)

Columna(s) que rompen nacional_2013_2015: ['resultado_fiscal_final_vc']


In [28]:
for nivel, id_t in [("municipal", "municipal_2001_2003"), ("provincial", "provincial_2001_2003")]:
    faltantes = columnas_nan(nivel, id_t, columnas_finales_por_nivel[nivel],paneles)
    print(f"{id_t}: NaN en -> {faltantes}")


municipal_2001_2003: NaN en -> ['emae_nivel_vc', 'emae_pendiente_vc', 'emae_volatilidad_vc']
provincial_2001_2003: NaN en -> ['emae_nivel_vc', 'emae_pendiente_vc', 'emae_volatilidad_vc']


In [29]:
for nivel in NIVELES:
    columnas_finales_por_nivel[nivel] = [c for c in columnas_finales_por_nivel[nivel] if not c.startswith("emae_")]  
columnas_finales_por_nivel["nacional"] = [c for c in columnas_finales_por_nivel["nacional"] if not c.startswith("resultado_fiscal_final_vc")]
datos_final = {}
for nivel in NIVELES:
    X, y = construir_Xy_final(nivel, columnas_finales_por_nivel[nivel], paneles, target="delta_voto_exit_total_pct")
    datos_final[nivel] = (X, y)
    print(f"{nivel}: N={len(y)}, P={X.shape[1]}")

municipal: N=12, P=21
provincial: N=12, P=21
nacional: N=7, P=20


### Paso 3 — LASSO por coordinate descent (implementación propia)

Formulación: `(1/2n)·‖y - Xβ‖² + α·‖β‖₁` (convención sklearn/glmnet). `X` estandarizada a mano (`ddof=0`), `y` centrada; intercepto = `media(y)`, no se penaliza.

`soft_threshold`: operador proximal de L1, da la selección de variables (coeficiente exactamente en cero si `|z| <= alpha`). `lasso_coordinate_descent`: actualiza una coordenada de `β` a la vez hasta que el cambio máximo entre iteraciones sea `< tol`.

In [30]:
for nivel in NIVELES:
    X_df, y_ser = datos_final[nivel]
    X_std, medias, desvios = estandarizar(X_df)
    y_centrado = y_ser.values - y_ser.mean()
    n = len(y_centrado)

    alpha_prueba = 1.0
    beta_manual = lasso_coordinate_descent(X_std, y_centrado, alpha_prueba)

    kkt = verificar_kkt(X_std, y_centrado, beta_manual, alpha_prueba, n)
    print(f"{nivel} - KKT:", kkt)

    # Chequeo 2: alpha=0 debe coincidir con OLS
    beta_alpha_cero = lasso_coordinate_descent(X_std, y_centrado, alpha=0.0, max_iter=5000)
    beta_ols, *_ = np.linalg.lstsq(X_std, y_centrado, rcond=None)
    print(f"{nivel} - Máxima diferencia vs. OLS (alpha=0):", np.max(np.abs(beta_alpha_cero - beta_ols)))

    residuo_manual = y_centrado - X_std @ beta_alpha_cero
    residuo_ols = y_centrado - X_std @ beta_ols

    print(f"{nivel} - Residuo manual (debería ser ~0 si el sistema es subdeterminado):", np.max(np.abs(residuo_manual)))
    print(f"{nivel} - Residuo OLS (debería ser ~0 también):", np.max(np.abs(residuo_ols)))

municipal - KKT: {'error_max_en_activos': np.float64(3.846455634004542e-07), 'exceso_max_en_inactivos': np.float64(-0.16137734892340383), 'n_activos': np.int64(5)}
municipal - Máxima diferencia vs. OLS (alpha=0): 1.4523597700848392
municipal - Residuo manual (debería ser ~0 si el sistema es subdeterminado): 5.0913436928645694e-06
municipal - Residuo OLS (debería ser ~0 también): 1.4210854715202004e-14
provincial - KKT: {'error_max_en_activos': np.float64(2.0801618350052564e-07), 'exceso_max_en_inactivos': np.float64(-0.013482435072452503), 'n_activos': np.int64(3)}


provincial - Máxima diferencia vs. OLS (alpha=0): 2.2267582921512923
provincial - Residuo manual (debería ser ~0 si el sistema es subdeterminado): 4.7280755222089965e-06
provincial - Residuo OLS (debería ser ~0 también): 1.4210854715202004e-14
nacional - KKT: {'error_max_en_activos': np.float64(4.899797608759471e-07), 'exceso_max_en_inactivos': np.float64(-0.05889270947545888), 'n_activos': np.int64(4)}
nacional - Máxima diferencia vs. OLS (alpha=0): 2.0586271131593836
nacional - Residuo manual (debería ser ~0 si el sistema es subdeterminado): 6.599062656320598e-07
nacional - Residuo OLS (debería ser ~0 también): 1.5987211554602254e-14


### Paso 4 — Grilla de alpha + LOO-CV manual

In [31]:
resultados_cv = {}
for nivel in NIVELES:
    X_df, y_ser = datos_final[nivel]
    resultados_cv[nivel] = lasso_loocv_manual(X_df, y_ser, factor_extension=3.0)
    r = resultados_cv[nivel]
    print(f"{nivel}: alpha_min={r['alpha_min']:.4f}  alpha_1se={r['alpha_1se']:.4f}  (techo grilla={r['alphas'][-1]:.4f})")

municipal: alpha_min=3.0118  alpha_1se=7.0173  (techo grilla=7.0173)
provincial: alpha_min=3.5291  alpha_1se=7.1414  (techo grilla=7.1414)
nacional: alpha_min=5.9415  alpha_1se=13.8434  (techo grilla=13.8434)


In [34]:
for nivel in NIVELES:
    X_df, y_ser = datos_final[nivel]
    print(f"--- {nivel} ---")
    print(verificar_saturacion(X_df, y_ser, factores=[1, 3, 10]))
    print()

--- municipal ---
   factor_extension      techo  alpha_min  alpha_1se    mse_min  mse_en_techo
0                 1   2.339090   2.339090   2.339090  22.949037     22.949037
1                 3   7.017271   3.011767   7.017271  22.295555     22.295555
2                10  23.390903   3.250155  23.390903  22.295555     22.295555

--- provincial ---
   factor_extension      techo  alpha_min  alpha_1se    mse_min  mse_en_techo
0                 1   2.380465   2.380465   2.380465  29.530279     29.530279
1                 3   7.141396   3.529073   7.141396  27.180469     27.180469
2                10  23.804653   3.307646  23.804653  27.180469     27.180469

--- nacional ---
   factor_extension      techo  alpha_min  alpha_1se    mse_min  mse_en_techo
0                 1   4.614467   4.614467   4.614467  50.528801     50.528801
1                 3  13.843401   5.941497  13.843401  49.382150     49.382150
2                10  46.144671   5.568705  46.144671  49.382150     49.382150



### Paso 6 — Ajuste final: coeficientes y mejora sobre baseline trivial

`baseline_trivial_loocv`: MSE en LOO de predecir el promedio de los demás puntos -- piso de comparación. `mse_en_alpha`: mismo esquema de LOO que `lasso_loocv_manual`, pero para un alpha puntual (se recalcula, no se guarda en la función de CV).

**Resultado:**

| Nivel | Mejora de `alpha_min` sobre baseline | Mejora de `alpha_1se` |
|---|---|---|
| Municipal | +40.7% | +13.3% |
| Provincial | +14.0% | +0.0% |
| Nacional | +0.0% | +0.0% |

Coeficientes en `alpha_min`: `icg_pendiente_vc` sobrevive en municipal (6.41) y provincial (4.41), `reservas_pendiente_vc` solo en municipal (0.96, débil). En `alpha_1se`: solo `icg_pendiente_vc` en municipal (2.81), nada en provincial ni nacional.

In [35]:
resumen = []
coeficientes_min, coeficientes_1se = {}, {}

for nivel in NIVELES:
    X_df, y_ser = datos_final[nivel]
    r = resultados_cv[nivel]

    base = baseline_trivial_loocv(y_ser)
    mse_min = mse_en_alpha(X_df, y_ser, r["alpha_min"])
    mse_1se = mse_en_alpha(X_df, y_ser, r["alpha_1se"])

    resumen.append({
        "nivel": nivel,
        "baseline_mse": base,
        "mejora_alpha_min_%": 100 * (1 - mse_min / base),
        "mejora_alpha_1se_%": 100 * (1 - mse_1se / base),
    })

    coeficientes_min[nivel] = ajustar_final(X_df, y_ser, r["alpha_min"])
    coeficientes_1se[nivel] = ajustar_final(X_df, y_ser, r["alpha_1se"])

tabla_resumen = pd.DataFrame(resumen).set_index("nivel")
print(tabla_resumen)

            baseline_mse  mejora_alpha_min_%  mejora_alpha_1se_%
nivel                                                           
municipal      22.295555                 0.0                 0.0
provincial     27.180469                 0.0                 0.0
nacional       49.382150                 0.0                 0.0


In [36]:
tabla_coef_min = pd.DataFrame(coeficientes_min)
tabla_coef_min = tabla_coef_min[(tabla_coef_min != 0).any(axis=1)]
print("Coeficientes distintos de cero (alpha_min):")
tabla_coef_min

Coeficientes distintos de cero (alpha_min):


,municipal,provincial,nacional
desocupacion_final_vc,NaN,NaN,0.0
desocupacion_pendiente_vc,NaN,NaN,0.0
desocupacion_volatilidad_vc,0.0,0.0,NaN
reservas_final_vc,0.0,0.0,NaN
reservas_pendiente_vc,0.0,0.0,NaN


### Chequeo de estabilidad (leave-one-transition-out) — los tres niveles

**Municipal (alpha_1se=7.71):** Signo estable (`icg_pendiente_vc` siempre positivo), magnitud sensible a `2009_2011` y `2011_2013` (caída a 0.48/0.78 vs. 2.3-3.7 en el resto). Composición casi estable (`reservas_pendiente_vc` se activa débilmente solo al sacar `2005_2007`).

**Provincial (alpha_min=5.45 -- alpha_1se no tuvo sobrevivientes):**
- Signo estable: `icg_pendiente_vc` positivo en las 12 corridas, nunca en cero.
- Magnitud más variable que en municipal: rango 1.41 (sacando `2009_2011`) a 5.55 (sacando `2005_2007`). **`2009_2011` vuelve a ser la ventana de mayor apalancamiento, igual que en municipal** -- coincidencia entre niveles que sugiere que esa transición puntual tiene un peso real en el vínculo confianza-voto.
- Composición menos estable que en municipal: sacar `2015_2017` activa `icc_pendiente_vc` (2.02) mientras `icg` cae a 2.18 -- acá sí cambia cuál variable "aporta", no solo cuánto. `reservas_pendiente_vc`/`reservas_volatilidad_vc` aparecen de forma esporádica y débil en varias otras corridas, sin patrón consistente -- lectura: ruido, no señal.

**Nacional (alpha_min=11.85):** ninguna variable sobrevive en ninguna de las 7 corridas -- resultado nulo robusto.

**Síntesis:** `icg_pendiente_vc` es el hallazgo más sólido del ejercicio de LASSO -- signo positivo y consistente en municipal y provincial, con la particularidad de que la transición `2009_2011` reduce su magnitud en ambos niveles simultáneamente. Provincial es menos estable en composición que municipal (coherente con su menor mejora sobre baseline, 14% vs. 40.7%). Nacional no tiene ninguna señal individual bajo ningún criterio.

In [ ]:
ALPHA_PARA_ESTABILIDAD = {
    "municipal": ("alpha_1se", resultados_cv["municipal"]["alpha_1se"]),
    "provincial": ("alpha_1se", resultados_cv["provincial"]["alpha_1se"]),
    "nacional": ("alpha_1se", resultados_cv["nacional"]["alpha_1se"]),
}

for nivel in NIVELES:
    df = paneles[nivel]
    criterio, alpha = ALPHA_PARA_ESTABILIDAD[nivel]
    X_df, y_ser = datos_final[nivel] 
    resultado = estabilidad_seleccion(nivel, alpha, df, columnas_finales_por_nivel[nivel], "delta_voto_exit_total_pct", X_df, y_ser)
    sobrevivientes = resultado.loc[:, (resultado != 0).any(axis=0)]

    print(f"--- {nivel} (alpha={alpha:.3f}, criterio={criterio}) ---")
    if sobrevivientes.empty:
        print("Ninguna variable sobrevive en ninguna de las corridas leave-one-transition-out.\n")
    else:
        print(sobrevivientes)
        print()

--- municipal (alpha=7.017, criterio=alpha_1se) ---
Ninguna variable sobrevive en ninguna de las corridas leave-one-transition-out.

--- provincial (alpha=3.529, criterio=alpha_min) ---
Ninguna variable sobrevive en ninguna de las corridas leave-one-transition-out.

--- nacional (alpha=5.941, criterio=alpha_min) ---
Ninguna variable sobrevive en ninguna de las corridas leave-one-transition-out.

